# Lineaarinen optimointi

**Optimoitava (max) funktio:** 
z = 250x + 320y 
=> -z = -250x - 320y

**Rajoittavat funktiot:**   
12x + 20y <= 25 000  
16x + 7y <= 30 000  
12x + 6y <= 30 000  
22x + 15y <= 20 000  
x >= 0   
y >= 0


In [142]:
# tuodaan kirjasto

from scipy.optimize import linprog

In [97]:
# luodaan matriisi lähtötietojen perusteella

obj = [-250, -320]
lhs_ineq = [
    [12, 20],
    [16, 7],
    [12, 6],
    [22, 15]
]

rhs_ineq = [
    25000,
    30000,
    30000,
    20000
]

In [98]:
# ratkaistaan optimointiongelma

opt = linprog(c=obj, A_ub=lhs_ineq, b_ub=rhs_ineq, integrality=[1, 1], method="highs")
opt

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: -405510.0
              x: [ 9.500e+01  1.193e+03]
            nit: 5
          lower:  residual: [ 9.500e+01  1.193e+03]
                 marginals: [ 0.000e+00  0.000e+00]
          upper:  residual: [       inf        inf]
                 marginals: [ 0.000e+00  0.000e+00]
          eqlin:  residual: []
                 marginals: []
        ineqlin:  residual: [ 0.000e+00  2.013e+04  2.170e+04  1.500e+01]
                 marginals: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00]
 mip_node_count: 1
 mip_dual_bound: -405510.0
        mip_gap: 0.0

In [143]:
#tulostetaan optimoidut tuotantomäärät 

print(opt.x)

[  95. 1193.]


In [100]:
# varmistetaan tulos

print(250 * opt.x[0] + 320 * opt.x[1])
print(-opt.fun)

405510.0
405510.0


In [101]:
# tarkistetaan myös rajoittavat funkiot lasketuilla optimiarvoilla 

print([opt.x[0] * ineq[0] + opt.x[1] * ineq[1] for ineq in lhs_ineq])

[np.float64(25000.0), np.float64(9871.0), np.float64(8298.0), np.float64(19985.0)]


In [102]:
# tulostetaan vielä sama raja-arvojen kanssa vertailun helpottamiseksi

for ineq, limit in zip(lhs_ineq, rhs_ineq):
    print(f"{opt.x[0] * ineq[0] + opt.x[1] * ineq[1]} <= {limit}")

25000.0 <= 25000
9871.0 <= 30000
8298.0 <= 30000
19985.0 <= 20000


Huomaamme, että työvaiheet yksi ja neljä saavuttavat nykyisessä tuotantomallissa kapasiteettinsa huomattavasti nopeammin kuin työvaiheet 2 ja 3.

In [104]:
import pulp 
print(pulp.__version__)
pulp.listSolvers(onlyAvailable=True)

4.0.0


['COIN_CMD']

In [144]:
# ratkaistaan sama ongelma pulp kirjastoa hyödyntäen

from pulp import LpMaximize, LpProblem, lpSum, LpVariable, COIN_CMD

In [141]:
# luodaan malli-olio

model = LpProblem(name="tuotantokapasiteetin-optimointi", sense=LpMaximize)

In [136]:
# tarkastellaan LpProblem-luokkaa 

dir(model)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_addConstraintFromDataclass',
 '_from_rust_model_copy',
 '_model',
 '_pick_solver',
 '_sense',
 '_sos_dicts_for_xpress',
 '_sos_lists_for_mps_dataclass',
 'add',
 'addConstraint',
 'add_sos_group',
 'add_variable',
 'add_variable_dict',
 'add_variable_dicts',
 'add_variable_matrix',
 'assignConsPi',
 'assignConsSlack',
 'assignVarsDj',
 'assignVarsVals',
 'checkDuplicateConstraints',
 'checkDuplicateVars',
 'checkLengthVars',
 'clear_sos',
 'coefficients',
 'constraints',
 'copy',
 'deepcopy',
 'dummyVar',
 'exported_variables',
 'fixObjective',


In [106]:
# alustetaan päätösmuuttujat

x = model.add_variable(name="hila", lowBound=0, cat="Integer")
y = model.add_variable(name="vitkutin", lowBound=0, cat="Integer")

In [107]:
# lisätään rajoitteet

model += (12 * x + 20 * y <= 25000, "leikkaus")
model += (16 * x + 7 * y <= 30000, "stanssaus")
model += (12 * x + 6 * y <= 30000, "puristus")
model += (22 * x + 15 * y <= 20000, "maalaus")

# ja tavoitefunktio
model += (250 * x + 320 * y, "katetuotto")

In [108]:
# tarkastellaan muodostettua mallia

model

tuotantokapasiteetin-optimointi:
MAXIMIZE
250.0*hila + 320.0*vitkutin + 0.0
SUBJECT TO
leikkaus: 12 hila + 20 vitkutin <= 25000

stanssaus: 16 hila + 7 vitkutin <= 30000

puristus: 12 hila + 6 vitkutin <= 30000

maalaus: 22 hila + 15 vitkutin <= 20000

VARIABLES
0 <= hila Integer
0 <= vitkutin Integer

In [109]:
# ratkaistaan
# tulosteesta voi lukea mm. käytetyn ratkaisualgoritmin (solver='COIN_CMD'), statuksen, ja tavoiteratkaisun

model.solve(COIN_CMD(msg=False))

LpSolveStats(solver='COIN_CMD', status=<LpSolveStatus.Optimal: 1>, has_solution=True, time=0.024966955184936523, cpu_time=0.015625, objective=405510.0, best_bound=None, num_variables=2, num_constraints=4, is_mip=True, solver_options={'solver': 'COIN_CMD', 'mip': True, 'msg': False, 'keepFiles': False, 'warmStart': False, 'timeMode': 'elapsed', 'logPath': 'C:\\Users\\iinak\\AppData\\Local\\Temp\\tmp817eina4-pulp.log'}, logs={'version': None, 'solver': 'CBC', 'status': None, 'best_bound': None, 'best_solution': None, 'gap': None, 'time': None, 'matrix_post': None, 'matrix': None, 'cut_info': None, 'rootTime': None, 'presolve': None, 'first_relaxed': None, 'progress': [], 'first_solution': None, 'status_code': None, 'sol_code': None, 'nodes': 0})

In [110]:
print(f"""Päätösmuuttujien arvot:
{[f"{var.name}: {var.value():.0f}" for var in model.variables()]}""")

Päätösmuuttujien arvot:
['hila: 95', 'vitkutin: 1193']


**Vastaukset tehtäviin 1 ja 2**

In [135]:
print(f"""Maksimikate: 
{model.solve().objective:.2f} euroa

Optimoidut tuontatomäärät: 
{model.variables()[0].value():.0f} hilaa ja {model.variables()[1].value():.0f} vitkutinta""")

print("\nKäyttämätön kapasiteetti:")

for constraint in model.constraints():
    print(f"{constraint.name}: {-constraint.value():.0f} min")

Maksimikate: 
405510.00 euroa

Optimoidut tuontatomäärät: 
95 hilaa ja 1193 vitkutinta

Käyttämätön kapasiteetti:
leikkaus: -0 min
stanssaus: 20129 min
puristus: 21702 min
maalaus: 15 min


In [114]:
# luodaan uusi malli parhaan investointikohteen ratkaisemiseksi, kun 
# yhteen työvaiheeseen voidaan tehdä 25 000 euron investointi, 
# jolla kapasiteettia kasvatetaan 25 000 minuutilla

model2 = LpProblem(name="resurssien-allokointi", sense=-1)

# luodaan muuttujat
stages = ["leikkaus", "stanssaus", "puristus", "maalaus"]
investment = {s: model2.add_variable(name=s, cat="Binary") for s in stages}

h = model2.add_variable(name="hila", lowBound=0, cat="Integer")
v = model2.add_variable(name="vitkutin", lowBound=0, cat="Integer")

model2.variables()

[leikkaus, stanssaus, puristus, maalaus, hila, vitkutin]

In [115]:
# lisätään rajoitteet
amount = 25000
model2 += (lpSum(investment.values()) == 1)
model2 += (12 * h + 20 * v <= 25000 + investment["leikkaus"] * amount, "leikkaus")
model2 += (16 * h + 7 * v <= 30000 + investment["stanssaus"] * amount, "stanssaus")
model2 += (12 * h + 6 * v <= 30000 + investment["puristus"] * amount, "puristus")
model2 += (22 * h + 15 * v <= 20000 + investment["maalaus"] * amount, "maalaus")

# ja tavoite
model2.setObjective(250 * h + 320 * v)

In [116]:
# tarkastellaan mallia

print(model2)

resurssien-allokointi:
MAXIMIZE
250.0*hila + 320.0*vitkutin + 0.0
SUBJECT TO
_C1: leikkaus + maalaus + puristus + stanssaus = 1

leikkaus: 12 hila - 25000 leikkaus + 20 vitkutin <= 25000

stanssaus: 16 hila - 25000 stanssaus + 7 vitkutin <= 30000

puristus: 12 hila - 25000 puristus + 6 vitkutin <= 30000

maalaus: 22 hila - 25000 maalaus + 15 vitkutin <= 20000

VARIABLES
0 <= leikkaus <= 1 Integer
0 <= stanssaus <= 1 Integer
0 <= puristus <= 1 Integer
0 <= maalaus <= 1 Integer
0 <= hila Integer
0 <= vitkutin Integer



In [117]:
# raktaistaan ongelma ja tarkastellaan ratkaisua 

solution2 = model2.solve(COIN_CMD(msg=False))
print(solution2)

solver: COIN_CMD
status: Optimal (1)
has solution: True
objective: 504400.0
time: 0.0283s (cpu 0.0000s)
nodes: 0


In [118]:
for constraint in model2.constraints():
    print(constraint.value())

0.0
0.0
-10.0
-7380.0
-2850.0


**Vastaus tehtävään 3**

In [124]:
# Tulostetaan optimoinnin tulokset
print("Optimoitu tuotantosuunnitelma:")
print(f"  Hila: {h.value():.0f} kpl")
print(f"  Vitkutin: {v.value():.0f} kpl")

print(f"\nMaksimikate: {solution2.objective:.0f} €")

print("\nInvestointi:")
for stage, variable in investment.items():
    if variable.value() == 1:
        print(f"  Investointikohde: {stage}")
        print(f"  Lisäkapasiteetti: {amount:.0f} min")
        print(f"  Investoitikustannukset: {amount:.0f} €")

print("\nKäyttämätön kapasiteetti:")
for constraint in model2.constraints()[1:]:
    print(f"  {constraint.name}: {-constraint.value():.0f} min")

Optimoitu tuotantosuunnitelma:
  Hila: 1800 kpl
  Vitkutin: 170 kpl

Maksimikate: 504400 €

Investointi:
  Investointikohde: maalaus
  Lisäkapasiteetti: 25000 min
  Investoitikustannukset: 25000 €

Käyttämätön kapasiteetti:
  leikkaus: -0 min
  stanssaus: 10 min
  puristus: 7380 min
  maalaus: 2850 min


In [120]:
# luodaan malli kuvaamaan kolmatta skenaarioa, jossa tuotannossa on yksi tuote lisää

model3 = LpProblem(sense=-1, name="tuotannon-optimointi")

x1 = model3.add_variable(name="hila", lowBound=0, cat="Integer")
x2 = model3.add_variable(name="vitkutin", lowBound=0, cat="Integer")
x3 = model3.add_variable(name="vilunki", lowBound=0, cat="Integer")

model3 += (12 * x1 + 20 * x2 + 30 * x3 <= 25000, "leikkaus")
model3 += (16 * x1 + 7 * x2 + 20 * x3 <= 30000, "stanssaus")
model3 += (12 * x1 + 6 * x2 + 10 * x3 <= 30000, "puristus")
model3 += (22 * x1 + 15 * x2 + 25 * x3 <= 20000, "maalaus")

model3.setObjective(250 * x1 + 320 * x2 + 350 * x3)

print(model3)

tuotannon-optimointi:
MAXIMIZE
250.0*hila + 350.0*vilunki + 320.0*vitkutin + 0.0
SUBJECT TO
leikkaus: 12 hila + 30 vilunki + 20 vitkutin <= 25000

stanssaus: 16 hila + 20 vilunki + 7 vitkutin <= 30000

puristus: 12 hila + 10 vilunki + 6 vitkutin <= 30000

maalaus: 22 hila + 25 vilunki + 15 vitkutin <= 20000

VARIABLES
0 <= hila Integer
0 <= vitkutin Integer
0 <= vilunki Integer



In [121]:
# ratkaistaan
Ssolution3 = model3.solve(COIN_CMD(msg=False))
print(solution3)

solver: COIN_CMD
status: Optimal (1)
has solution: True
objective: 405510.0
time: 0.0252s (cpu 0.0000s)
nodes: 0


**Vastaus tehtävään 4**

In [125]:
# Tulostetaan optimoinnin tulokset
print("Optimoitu tuotantosuunnitelma:")
print(f"  Hila: {x1.value():.0f} kpl")
print(f"  Vitkutin: {x2.value():.0f} kpl")
print(f"  Vilunki: {x3.value():.0f} kpl")

print(f"\nMaksimikate: {solution3.objective:.0f} €")

print("\nKäyttämätön kapasiteetti:")
for constraint in model3.constraints():
    print(f"  {constraint.name}: {-constraint.value():.0f} min")

Optimoitu tuotantosuunnitelma:
  Hila: 95 kpl
  Vitkutin: 1193 kpl
  Vilunki: 0 kpl

Maksimikate: 405510 €

Käyttämätön kapasiteetti:
  leikkaus: -0 min
  stanssaus: 20129 min
  puristus: 21702 min
  maalaus: 15 min
